# Transposição: system.lakeflow → observabilidade_runs/tasks

## Objetivo
Espelhar runs e tasks do `system.lakeflow` para tabelas UC próprias, com retenção controlada e RBAC.

## Lógica
1. Watermark: `MAX(period_start_time)` de cada tabela
2. MERGE incremental `job_run_timeline → observabilidade_runs` (CAST STRING→BIGINT, dedup via QUALIFY)
3. MERGE incremental `job_task_run_timeline → observabilidade_tasks` (CAST STRING→BIGINT, dedup via QUALIFY)
4. Filtro de ambiente: INNER JOIN com `jobs_metadata` (já filtrada por ambiente)

## Idempotência
MERGE por `run_id`. Re-execução não duplica. Watermark avança a cada execução.

In [0]:
%run ./config_parametros

In [0]:
spark.sql(f"""
MERGE INTO {SCHEMA_APOIO}.observabilidade_runs AS t
USING (
  WITH watermark AS (
    SELECT COALESCE(MAX(period_start_time), TIMESTAMP '1970-01-01') AS wm
    FROM {SCHEMA_APOIO}.observabilidade_runs
  ),
  runs_incremental AS (
    SELECT
      CAST(run_id AS BIGINT)      AS run_id,
      CAST(job_id AS BIGINT)      AS job_id,
      workspace_id,
      trigger_type,
      result_state,
      run_type,
      period_start_time,
      period_end_time,
      execution_duration_seconds,
      run_duration_seconds
    FROM system.lakeflow.job_run_timeline
    WHERE workspace_id = '7474657818873516'
      AND period_start_time >= (SELECT wm FROM watermark)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY run_id ORDER BY period_end_time DESC) = 1
  )
  SELECT
    r.run_id,
    r.job_id,
    r.workspace_id,
    r.trigger_type,
    r.result_state,
    r.run_type,
    r.period_start_time,
    r.period_end_time,
    r.execution_duration_seconds,
    r.run_duration_seconds,
    j.job_name,
    j.ambiente,
    current_timestamp() AS ingested_at
  FROM runs_incremental r
  INNER JOIN {SCHEMA_APOIO}.jobs_metadata j ON r.job_id = j.job_id
) AS s
ON t.run_id = s.run_id
WHEN MATCHED THEN UPDATE SET
  job_id = s.job_id,
  workspace_id = s.workspace_id,
  trigger_type = s.trigger_type,
  result_state = s.result_state,
  run_type = s.run_type,
  period_start_time = s.period_start_time,
  period_end_time = s.period_end_time,
  execution_duration_seconds = s.execution_duration_seconds,
  run_duration_seconds = s.run_duration_seconds,
  job_name = s.job_name,
  ambiente = s.ambiente,
  ingested_at = s.ingested_at
WHEN NOT MATCHED THEN INSERT (
  run_id, job_id, workspace_id, trigger_type, result_state, run_type,
  period_start_time, period_end_time, execution_duration_seconds,
  run_duration_seconds, job_name, ambiente, ingested_at
)
VALUES (
  s.run_id, s.job_id, s.workspace_id, s.trigger_type, s.result_state, s.run_type,
  s.period_start_time, s.period_end_time, s.execution_duration_seconds,
  s.run_duration_seconds, s.job_name, s.ambiente, s.ingested_at
)
""")

print("✅ observabilidade_runs: MERGE concluído")

In [0]:
spark.sql(f"""
MERGE INTO {SCHEMA_APOIO}.observabilidade_tasks AS t
USING (
  WITH watermark AS (
    SELECT COALESCE(MAX(period_start_time), TIMESTAMP '1970-01-01') AS wm
    FROM {SCHEMA_APOIO}.observabilidade_tasks
  ),
  tasks_incremental AS (
    SELECT
      CAST(run_id AS BIGINT)        AS run_id,
      CAST(job_run_id AS BIGINT)    AS job_run_id,
      CAST(job_id AS BIGINT)        AS job_id,
      task_key,
      workspace_id,
      result_state,
      period_start_time,
      period_end_time,
      execution_duration_seconds,
      setup_duration_seconds,
      cleanup_duration_seconds,
      termination_code
    FROM system.lakeflow.job_task_run_timeline
    WHERE workspace_id = '7474657818873516'
      AND period_start_time >= (SELECT wm FROM watermark)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY run_id ORDER BY period_end_time DESC) = 1
  )
  SELECT
    ti.run_id,
    ti.job_run_id,
    ti.task_key,
    ti.workspace_id,
    ti.result_state,
    ti.period_start_time,
    ti.period_end_time,
    ti.execution_duration_seconds,
    ti.setup_duration_seconds,
    ti.cleanup_duration_seconds,
    ti.termination_code,
    current_timestamp() AS ingested_at
  FROM tasks_incremental ti
  INNER JOIN {SCHEMA_APOIO}.jobs_metadata j ON ti.job_id = j.job_id
) AS s
ON t.run_id = s.run_id
WHEN MATCHED THEN UPDATE SET
  job_run_id = s.job_run_id,
  task_key = s.task_key,
  workspace_id = s.workspace_id,
  result_state = s.result_state,
  period_start_time = s.period_start_time,
  period_end_time = s.period_end_time,
  execution_duration_seconds = s.execution_duration_seconds,
  setup_duration_seconds = s.setup_duration_seconds,
  cleanup_duration_seconds = s.cleanup_duration_seconds,
  termination_code = s.termination_code,
  ingested_at = s.ingested_at
WHEN NOT MATCHED THEN INSERT (
  run_id, job_run_id, task_key, workspace_id, result_state,
  period_start_time, period_end_time, execution_duration_seconds,
  setup_duration_seconds, cleanup_duration_seconds, termination_code, ingested_at
)
VALUES (
  s.run_id, s.job_run_id, s.task_key, s.workspace_id, s.result_state,
  s.period_start_time, s.period_end_time, s.execution_duration_seconds,
  s.setup_duration_seconds, s.cleanup_duration_seconds, s.termination_code, s.ingested_at
)
""")

print("✅ observabilidade_tasks: MERGE concluído")

In [0]:
print("=== observabilidade_runs ===")
spark.sql(f"SELECT COUNT(*) AS total, COUNT(DISTINCT run_id) AS distinct_runs FROM {SCHEMA_APOIO}.observabilidade_runs").show()
spark.sql(f"SELECT result_state, COUNT(*) FROM {SCHEMA_APOIO}.observabilidade_runs GROUP BY result_state").show()
spark.sql(f"SELECT job_name, ambiente, COUNT(*) AS runs FROM {SCHEMA_APOIO}.observabilidade_runs GROUP BY job_name, ambiente ORDER BY runs DESC").show()

print("=== observabilidade_tasks ===")
spark.sql(f"SELECT COUNT(*) AS total, COUNT(DISTINCT run_id) AS distinct_tasks FROM {SCHEMA_APOIO}.observabilidade_tasks").show()
spark.sql(f"SELECT result_state, COUNT(*) FROM {SCHEMA_APOIO}.observabilidade_tasks GROUP BY result_state").show()

print("=== Sanity: runs com job_name NULL ===")
spark.sql(f"SELECT COUNT(*) AS null_job_names FROM {SCHEMA_APOIO}.observabilidade_runs WHERE job_name IS NULL").show()